In [1]:
import sys
from pathlib import Path
import cobra

In [2]:
COFACTORES = [
    "h2o", "h", "pi", "ppi", "co2", "nh4",
    "atp", "adp", "amp",
    "nad", "nadh", "nadp", "nadph",
    "coa", "accoa",
    "fad", "fadh2",
    "nadph", "thf", "amet", "ahcys",
    "q8", "q8h2",
]

In [3]:
def normaliza(met_id: str, compartimento: str = "e") -> str:
    """Devuelve el ID en el formato que espera Seed2LP: M_<id>_<comp>.
 
    El README es explicito: cada metabolito debe llevar prefijo 'M_' para
    coincidir con el ID del SBML. Si el ID ya viene con compartimento, se
    respeta; si no, se le anade.
    """
    mid = met_id.strip()
    if not mid:
        return ""
    if mid.startswith("M_"):
        return mid
    if not mid.endswith(f"_{compartimento}"):
        mid = f"{mid}_{compartimento}"
    return f"M_{mid}"

In [4]:
def ids_del_modelo(modelo) -> set:
    """IDs de metabolitos realmente presentes, ya normalizados."""
    return {normaliza(m.id) for m in modelo.metabolites}

In [5]:
def escribe(ruta: Path, elementos) -> None:
    ruta.write_text("\n".join(sorted(set(elementos))) + "\n")
    print(f"  {ruta.name}: {len(set(elementos))} entradas")

In [10]:
def main(modelo_sbml, outdir, dieta_txt=None):
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    modelo = cobra.io.read_sbml_model(modelo_sbml)
    presentes = ids_del_modelo(modelo)
    print(f"Modelo {modelo.id}: {len(modelo.metabolites)} metabolitos, "
          f"{len(modelo.reactions)} reacciones")

    if dieta_txt:
        dieta = [normaliza(l) for l in Path(dieta_txt).read_text().splitlines()
                 if l.strip() and not l.startswith("#")]
    else:
        dieta = []

    # IDs reales verificados contra AGORA2 v2.01
    FARMACO        = ["levdopa"]
    INTERMEDIARIO  = ["dpmn"]
    PRODUCTO_FINAL = ["mtym"]

    farmaco    = [normaliza(m) for m in FARMACO]
    prohibidos = [normaliza(m) for m in INTERMEDIARIO + PRODUCTO_FINAL]
    targets    = [normaliza(m) for m in PRODUCTO_FINAL]

    print("\nChequeo de IDs contra el modelo:")
    for nombre, lista in [("dieta", dieta),
                          ("cofactores", [normaliza(c) for c in COFACTORES]),
                          ("farmaco", farmaco),
                          ("prohibidos", prohibidos),
                          ("targets", targets)]:
        faltan = [m for m in lista if m not in presentes]
        estado = "OK" if not faltan else f"FALTAN {len(faltan)}: {faltan[:5]}"
        print(f"  {nombre:12s} {len(lista):4d} ids  -> {estado}")

    def filtra(lista):
        return [m for m in lista if m in presentes]

    seeds_known = filtra(dieta + [normaliza(c) for c in COFACTORES] + farmaco)
    extracelulares = [normaliza(m.id) for m in modelo.metabolites
                      if m.compartment in ("e", "C_e", "extracellular")]
    possible = [m for m in extracelulares if m not in prohibidos]

    print("\nArchivos generados:")
    escribe(outdir / "seeds_known.txt", seeds_known)
    escribe(outdir / "possible_seeds.txt", possible)
    escribe(outdir / "forbidden_seeds.txt", filtra(prohibidos))
    escribe(outdir / "targets.txt", filtra(targets))

In [11]:
main("sbml/Enterococcus_faecalis_ATCC_29212_AGORA2_v2.01.xml", "inputs")

Set parameter Username
Set parameter LicenseID to value 2854521
Academic license - for non-commercial use only - expires 2027-08-19
Modelo microbe_3280: 1486 metabolitos, 1560 reacciones

Chequeo de IDs contra el modelo:
  dieta           0 ids  -> OK
  cofactores     23 ids  -> FALTAN 15: ['M_atp_e', 'M_adp_e', 'M_amp_e', 'M_nad_e', 'M_nadh_e']
  farmaco         1 ids  -> OK
  prohibidos      2 ids  -> FALTAN 1: ['M_mtym_e']
  targets         1 ids  -> FALTAN 1: ['M_mtym_e']

Archivos generados:
  seeds_known.txt: 9 entradas
  possible_seeds.txt: 173 entradas
  forbidden_seeds.txt: 1 entradas
  targets.txt: 0 entradas


In [12]:
main("sbml/Eggerthella_lenta_A2_AGORA2_v2.01.xml", "inputs_lenta")

Modelo microbe_3170: 1170 metabolitos, 1143 reacciones

Chequeo de IDs contra el modelo:
  dieta           0 ids  -> OK
  cofactores     23 ids  -> FALTAN 17: ['M_atp_e', 'M_adp_e', 'M_amp_e', 'M_nad_e', 'M_nadh_e']
  farmaco         1 ids  -> FALTAN 1: ['M_levdopa_e']
  prohibidos      2 ids  -> OK
  targets         1 ids  -> OK

Archivos generados:
  seeds_known.txt: 6 entradas
  possible_seeds.txt: 99 entradas
  forbidden_seeds.txt: 2 entradas
  targets.txt: 1 entradas
